In [ ]:
# import packages
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel ('../data/Data.xlsx')
df.head()

In [ ]:
df_new = pd.read_excel ('../data/Daten_Siburg.xlsx')
df_new.head()

In [ ]:
plt.hist(df['h'], bins = 20, stacked = 'true', density = 'true')
plt.xlabel('h [mm]')
plt.show()

In [ ]:
EC_mean = 0.84
EC_std = 0.232
SVR_mean = 1.11
SVR_std = 0.55
LR_mean = 1.006
LR_std = 1.004
Tree_mean = 1.16
Tree_std = 0.52

plt.scatter((1,1,1),(EC_mean, EC_mean+EC_std, EC_mean-EC_std), color = 'tab:orange')
plt.scatter((2,2,2),(SVR_mean, SVR_mean+SVR_std, SVR_mean-SVR_std), color = 'm')
plt.scatter((3,3,3),(LR_mean, LR_mean+LR_std, LR_mean-LR_std), color = 'b')
plt.scatter((4,4,4),(Tree_mean, Tree_mean+Tree_std, Tree_mean-Tree_std), color = 'g')
plt.plot((1,4),(1,1), 'k--')
plt.plot((1,1),(EC_mean+EC_std,EC_mean-EC_std), color = 'tab:orange')
plt.plot((2,2),(SVR_mean+SVR_std,SVR_mean-SVR_std), 'm')
plt.plot((3,3),(LR_mean+LR_std,LR_mean-LR_std),'b')
plt.plot((4,4),(Tree_mean+Tree_std,Tree_mean-Tree_std),'g')
plt.scatter(1,1.3, color = 'tab:orange', marker='x')
plt.scatter(2,0.7, color='m', marker='x')
plt.scatter(3,2.5, color='b', marker='x')
plt.scatter(4,3.4, color='g', marker='x')
plt.xticks((1,2,3,4),labels=('EC','SVR','NLR','RF'))
plt.show()
            

In [ ]:
from sklearn.svm import SVR
import matplotlib.pyplot as plt

In [ ]:
# grid search hyperparameters
from numpy import arange
from sklearn.model_selection import GridSearchCV
# define model
model = SVR()
# define model evaluation method
#cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# define grid
grid = dict()
grid['C'] = arange(0, 50, 1)
#grid['gamma'] = arange(0, 0.01, 0.0001)
# define search
search = GridSearchCV(model, grid, cv=10, scoring='neg_mean_absolute_error', n_jobs=-1)
# perform the search
results = search.fit(X, y)
#print(results)

r = search.cv_results_
#print(search.cv_results_)

# summarize
print('MAE: %.3f' % results.best_score_)
print('Config: %s' % results.best_params_)

res = list(r.values())
len(res[6])
#plt.plot(arange(0, 50, 1), res[6], label='1')
plt.plot(arange(0, 50, 1), res[11], label='5')
plt.plot(arange(0, 50, 1), res[16], label='10')
#plt.plot(arange(0, 50, 1), res[21], label='15')
#plt.plot(arange(0, 50, 1), res[26], label='20')
plt.legend()
plt.xlabel('C')
plt.ylabel('MAE')
plt.show()

In [ ]:
# grid search hyperparameters
from numpy import arange
from sklearn.model_selection import GridSearchCV
# define model
model = SVR()
# define model evaluation method
#cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# define grid
grid = dict()
#grid['C'] = arange(0, 50, 1)
grid['gamma'] = arange(0, 0.0001, 0.0000001)
# define search
search = GridSearchCV(model, grid, cv=20, scoring='neg_mean_absolute_error', n_jobs=-1)
# perform the search
results = search.fit(X, y)
#print(results)

r = search.cv_results_
#print(search.cv_results_)

# summarize
print('MAE: %.3f' % results.best_score_)
print('Config: %s' % results.best_params_)

res = list(r.values())
plt.plot(arange(0, 0.0001, 0.0000001), res[11], label='5')
plt.plot(arange(0, 0.0001, 0.0000001), res[16], label='10')
plt.plot(arange(0, 0.0001, 0.0000001), res[21], label='15')
plt.plot(arange(0, 0.0001, 0.0000001), res[26], label='20')
plt.legend()
plt.xlabel('gamma')
plt.ylabel('MAE')
plt.show()

In [ ]:
MAE: -0.104
Config: {'gamma': 1.13e-05}
    
MAE: -0.100
Config: {'gamma': 1.08e-05}
    
MAE: -0.102
Config: {'gamma': 3.2999999999999997e-06}

In [ ]:
df = pd.read_excel ('../data/Data.xlsx')
df['V_test']

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# select the properties that should be scaled
cols=['d', 'Fläche', 'rho_l', 'fcm,cyl', 'V_test', 'V_Rd', 'V_adj']

# Create Scaler object
scaler = MinMaxScaler(feature_range=(0.000001, 1))
# fit scaler to data
scaler.fit(df[cols])
#scale chosen properties
df_scaled = pd.DataFrame(scaler.transform(df[cols]), columns=cols)

# inspect distribution
print(df_scaled)


In [ ]:
cols=['d', 'Fläche', 'rho_l', 'fcm,cyl', 'V_test', 'V_Rd', 'V_adj']
scaler.fit(df[cols])
df_rescaled = pd.DataFrame(scaler.inverse_transform(df_scaled), columns=cols)
df_rescaled

In [ ]:
V_adj = df['V_Rd']*1.5/0.18*0.229 #changing SF

plt.scatter(df['V_Rd'], df['V_test'], label='EC')
plt.scatter(V_adj, df['V_test'], label='EC_adjusted')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,3], [0,3], 'r--', label='V_test = V_predicted')
plt.plot([0,2], [0,2*1.8], 'b--', label='SF = 1.8')
plt.legend()
plt.show()
MSE = np.mean(np.square(np.subtract(df['V_test'], V_adj)))
print('MSE = ', MSE)

V_adj_scaled= df_scaled['V_Rd']*1.5/0.18*0.16
plt.scatter(df['V_adj_scaled'], df_scaled['V_test'], label='EC_adjusted_scaled', color = 'tab:orange')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'r--', label='V_test = V_predicted')
plt.legend()
MSE = np.mean(np.square(np.subtract(df_scaled['V_test'], df['V_adj_scaled'])))
plt.show()
print('MSE = ', MSE)

In [ ]:
plt.scatter(df['V_Rd'], df['V_test'], label='EC')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,2], [0,2*1.8], 'b--', label='SF = 1.8')
plt.legend()
plt.show()

In [ ]:
X = df['V_adj_scaled']
y = df_scaled['V_test']
EC = scaled_data[:,5]

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

V_adj_scaled = df_scaled['V_Rd']*1.5/0.18*0.16
plt.scatter(X_test, y_test, label='EC_adjusted_scaled', color = 'tab:orange')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'r--', label='V_test = V_predicted')
plt.legend()
MSE = np.mean(np.square(np.subtract(df_scaled['V_test'], df['V_adj_scaled'])))
plt.show()
print('MSE = ', MSE)

In [ ]:
print(np.mean(np.divide(X_test, y_test)))
print(np.std(np.divide(X_test, y_test)))
np.divide(df['V_adj_scaled'].values, df['V_test_scaled'].values)[249]

In [ ]:
#print(df['V_adj_scaled'][249])
#print(df['V_test'][249])
print(np.mean(np.divide(np.delete(df['V_adj_scaled'].values, 249), np.delete(df['V_test_scaled'].values, 249))))
print(np.std(np.divide(np.delete(df['V_adj_scaled'].values, 249), np.delete(df['V_test_scaled'].values, 249))))

In [ ]:
df['V_adj_scaled']

In [ ]:
from sklearn.model_selection import train_test_split
scaled_data = df_scaled.values
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

svr_rbf = SVR(kernel="rbf", C=6.15, gamma=1.2, epsilon=0.03)
svr_rbf.fit(X_train, y_train)
y_predict = svr_rbf.predict(X_test)
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
plt.scatter(y_predict, y_test, label='actual')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'k--')
plt.show()

MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
print('MSE = ', MSE)
print('#SV = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape[0])
print('percentage of SV w.r.t. dataset = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape[0]/X_train.shape[0])
#svr_rbf.fit(X_train, y_train).dual_coef_

fig = plt.figure(figsize=(17, 5))
ax1 = fig.add_subplot(131)
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

#fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.suptitle('rbf kernel')
ax1.scatter(y_predict, y_test, label='actual')
ax1.set(xlabel='V_predicted', ylabel='V_test')
ax1.plot([0,1], [0,1], 'k--')

ax2.scatter(X_train[:,0],X_train[:,1], label='data', c='g')
ax2.scatter(svr_rbf.fit(X_train, y_train).support_vectors_[:,0], svr_rbf.fit(X_train, y_train).support_vectors_[:,1], marker='x', label='SV', c='r')
ax2.set(xlabel='d', ylabel='Fläche')

ax3.scatter(X_train[:,2],X_train[:,3], label='data', c='g')
ax3.scatter(svr_rbf.fit(X_train, y_train).support_vectors_[:,2], svr_rbf.fit(X_train, y_train).support_vectors_[:,3], marker='x', label='SV', c='r')
ax3.set(xlabel='rho_l', ylabel='fcm_cyl')
plt.legend()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
scaled_data = df_scaled.values
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

svr_rbf = SVR(kernel="rbf", C=6.15, gamma=1.2, epsilon=0.03)
svr_rbf.fit(X_train, y_train)
y_predict = svr_rbf.predict(X_test)
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
plt.scatter(y_predict, y_test, label='actual')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'k--')
plt.show()
MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
print('MSE = ', MSE)
print('#SV = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape[0])

In [ ]:
from sklearn.model_selection import train_test_split
scaled_data = df_scaled.values
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

v_max=2.68
v_min=0.025

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

df_train_V_Rd, df_test_V_Rd = train_test_split(df['V_Rd'].values, stratify=y_binned, test_size=0.3,random_state=88)

svr_rbf = SVR(kernel="rbf", C=6.15, gamma=1.2, epsilon=0.03)
svr_rbf.fit(X_train, y_train)
y_predict = svr_rbf.predict(X_test)
y_rescaled = y_predict*(v_max-v_min)+v_min
y_EC_rescaled = df_test_V_Rd
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
#plt.scatter(y_rescaled, y_EC_rescaled, label='actual')
#plt.scatter(df['V_Rd']*1.5, df['V_test'])
plt.scatter(df_test_V_Rd*1.5, y_rescaled, color='b', label='EuroCode')
plt.scatter(df_test_V_Rd*1.5, y_rescaled/1.27, color='g', label="V_test_split SF=1.27")
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_Rd_Code [MN]')
plt.ylabel('V_predicted [MN]')
plt.plot([0,2.5], [0,2.5], 'k-')
plt.grid(True,linestyle='-',color='0.75')
plt.legend()
plt.show()


In [ ]:
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

#df_train_V_Rd, df_test_V_Rd = train_test_split(df, test_size=0.3,random_state=88)

v_max=2.68
v_min=0.025

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=88)
df_train_V_Rd, df_test_V_Rd = train_test_split(df['V_test'].values, test_size=0.3,random_state=88)

len(y_test)
len(df_test_V_Rd)

In [ ]:
#df.drop([ 'Platte', 'Last', 'Stütze', 'c2', 'Esm', 'c1','dg','h', 'Lasteinleitung', 'fym'], axis = 1, inplace=True)
#print(df)
Max=df.max()
Min=df.min()
#print(Max)
#print(Min)
v_max=2.68
v_min=0.025

#ypred = yhat_1*(v_max-v_min)+v_min
df['V_test'].values
df.head()

In [ ]:
print(np.mean(np.divide(np.delete(y_predict, 72), np.delete(y_test, 72))))
print(np.std(np.divide(np.delete(y_predict, 72), np.delete(y_test, 72))))

#np.std(np.square(np.subtract(y_test, y_predict)))

In [ ]:
from numpy import arange
from sklearn.model_selection import GridSearchCV
# define model
model = SVR(kernel='rbf')
# define model evaluation method
#cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# define grid
grid = dict()
grid['C'] = arange(1, 20, 10)
grid['gamma'] = np.linspace(0.01, 1, 10)
grid['epsilon'] = np.linspace(0.01, 0.1, 10)
# define search
search = GridSearchCV(model, grid, scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)
# perform the search
results = search.fit(X, y)
print(results)
# summarize
print('MAE: %.3f' % results.best_score_)
print('Config: %s' % results.best_params_)

In [ ]:
from sklearn.model_selection import train_test_split
scaled_data = df_scaled.values
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

svr_rbf = SVR(kernel="rbf", C=11, gamma=0.67, epsilon=0.02)
svr_rbf.fit(X_train, y_train)
y_predict = svr_rbf.predict(X_test)
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
plt.scatter(y_predict, y_test, label='actual')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'k--')
plt.show()
MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
print('MSE = ', MSE)
print('#SV = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape[0])

In [ ]:
plt.scatter(X_train[:,0],X_train[:,1], label='data', c='g')
plt.scatter(svr_rbf.fit(X_train, y_train).support_vectors_[:,0], svr_rbf.fit(X_train, y_train).support_vectors_[:,1], marker='x', label='SV', c='r')
plt.xlabel('d')
plt.ylabel('Fläche')
plt.legend()
plt.show()

plt.scatter(X_train[:,2],X_train[:,3], label='data', c='g')
plt.scatter(svr_rbf.fit(X_train, y_train).support_vectors_[:,2], svr_rbf.fit(X_train, y_train).support_vectors_[:,3], marker='x', label='SV', c='r')
plt.xlabel('rho_l')
plt.ylabel('fcm_cyl')
plt.legend()
plt.show()

In [ ]:
# search for smallest amount of SV
for c in np.linspace(0.1,2,20):
    svr_rbf = SVR(kernel="rbf", C=6.15, gamma=c, epsilon=0.03)
    svr_rbf.fit(X_train, y_train)
    y_predict = svr_rbf.predict(X_test)

    MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
    print('C =', c)
    print('MSE = ', MSE)
    print('SV = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape)

In [ ]:
C = 36
MSE =  0.00048082864475355717
SV =  (168, 4)
gamma = 0.9020.8

In [ ]:
from sklearn.model_selection import train_test_split
scaled_data = df_scaled.values
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

svr_poly = SVR(kernel="poly", degree=3, C=18, gamma=1.12, epsilon=0.07)
#svr_poly = SVR(kernel="poly", degree=3, C=15, gamma=0.73, epsilon=0.04)
svr_poly.fit(X_train, y_train)
y_predict = svr_poly.predict(X_test)
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
plt.scatter(y_predict, y_test, label='actual')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'k--')
plt.show()

MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
print('MSE = ', MSE)
print('#SV = ', svr_poly.fit(X_train, y_train).support_vectors_.shape[0])
#print('percentage of SV w.r.t. dataset = ', svr_poly.fit(X_train, y_train).support_vectors_.shape[0]/X_train.shape[0])
fig = plt.figure(figsize=(17, 5))
ax1 = fig.add_subplot(131)
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

#fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.suptitle('polynomial kernel')
ax1.scatter(y_predict, y_test, label='actual')
ax1.set(xlabel='V_predicted', ylabel='V_test')
ax1.plot([0,1], [0,1], 'k--')

ax2.scatter(X_train[:,0],X_train[:,1], label='data', c='g')
ax2.scatter(svr_poly.fit(X_train, y_train).support_vectors_[:,0], svr_poly.fit(X_train, y_train).support_vectors_[:,1], marker='x', label='SV', c='r')
ax2.set(xlabel='d', ylabel='Fläche')

ax3.scatter(X_train[:,2],X_train[:,3], label='data', c='g')
ax3.scatter(svr_poly.fit(X_train, y_train).support_vectors_[:,2], svr_poly.fit(X_train, y_train).support_vectors_[:,3], marker='x', label='SV', c='r')
ax3.set(xlabel='rho_l', ylabel='fcm_cyl')
plt.legend()
plt.show()

In [ ]:


from numpy import arange
from sklearn.model_selection import GridSearchCV
# define model
model = SVR(kernel='poly', degree=3)
# define model evaluation method
#cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# define grid
grid = dict()
grid['C'] = arange(15, 25, 10)
grid['gamma'] = np.linspace(0.1, 2, 10)
grid['epsilon'] = np.linspace(0.01, 0.1, 10)
# define search
search = GridSearchCV(model, grid, scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)
# perform the search
results = search.fit(X, y)
print(results)
# summarize
print('MAE: %.3f' % results.best_score_)
print('Config: %s' % results.best_params_)

In [ ]:
# search for smallest amount of SV
for c in np.linspace(0.01, 0.1, 10):
    svr_rbf = SVR(kernel="poly", C=18, gamma=1.12, epsilon=c)
    svr_rbf.fit(X_train, y_train)
    y_predict = svr_rbf.predict(X_test)

    MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
    print('C =', c)
    print('MSE = ', MSE)
    print('SV = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape)

In [ ]:
from sklearn.model_selection import train_test_split
scaled_data = df_scaled.values
X = scaled_data[:,0:4]
y = scaled_data[:,4]
EC = scaled_data[:,5]

# create categorical variables to stratify w.r.t y
bins = np.linspace(0, 1, 6)
y_binned = np.digitize(y, bins)

# split data (70% training set; 30% test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_binned, test_size=0.3, random_state=88)

#svr_lin = SVR(kernel="linear", C=0.21, gamma=0.8, epsilon=0.05)
svr_lin = SVR(kernel="linear", C=0.7)
svr_lin.fit(X_train, y_train)
y_predict = svr_lin.predict(X_test)
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
plt.scatter(y_predict, y_test, label='actual')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'k--')
plt.show()

MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
print('MSE = ', MSE)
print('#SV = ', svr_lin.fit(X_train, y_train).support_vectors_.shape[0])
print('percentage of SV w.r.t. dataset = ', svr_lin.fit(X_train, y_train).support_vectors_.shape[0]/X_train.shape[0])
print('feature importance ', svr_lin.fit(X_train, y_train).coef_)

fig = plt.figure(figsize=(17, 5))
ax1 = fig.add_subplot(131)
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

#fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.suptitle('linear kernel')
ax1.scatter(y_predict, y_test, label='actual')
ax1.set(xlabel='V_predicted', ylabel='V_test')
ax1.plot([0,1], [0,1], 'k--')

ax2.scatter(X_train[:,0],X_train[:,1], label='data', c='g')
ax2.scatter(svr_lin.fit(X_train, y_train).support_vectors_[:,0], svr_lin.fit(X_train, y_train).support_vectors_[:,1], marker='x', label='SV', c='r')
ax2.set(xlabel='d', ylabel='Fläche')

ax3.scatter(X_train[:,2],X_train[:,3], label='data', c='g')
ax3.scatter(svr_lin.fit(X_train, y_train).support_vectors_[:,2], svr_lin.fit(X_train, y_train).support_vectors_[:,3], marker='x', label='SV', c='r')
ax3.set(xlabel='rho_l', ylabel='fcm_cyl')
plt.legend()
plt.show()

In [ ]:
# search for smallest amount of SV

for g in np.linspace(.01, 5 ,20):
    svr_rbf = SVR(kernel="linear", C=0.7, epsilon=g)
    svr_rbf.fit(X_train, y_train)
    y_predict = svr_rbf.predict(X_test)

    MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
    print('gamma', g)
    print('MSE = ', MSE)
    print('SV = ', svr_rbf.fit(X_train, y_train).support_vectors_.shape)
    print('---------------------')

In [ ]:
#MAE: -0.016
#Config: {'C': 4.9, 'epsilon': 0.001, 'gamma': 0.9}

svr_rbf = SVR(kernel="rbf", C=7.4, gamma=0.902, epsilon=0.0041)
svr_rbf.fit(X_train, y_train)
y_predict = svr_rbf.predict(X_test)
x_axis = list(range(len(y_predict)))

#plt.scatter(x_axis, y_predict, label='prediction')
#plt.scatter(x_axis, y_test, label='actual')
plt.scatter(y_predict, y_test, color='b', label='SVM')
plt.scatter(EC, y, color='g', label = 'EC')
#plt.legend()
#plt.xlabel('datapoint')
plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.plot([0,1], [0,1], 'b--', label='V_test = V_predicted')
plt.plot([0,0.8], [0,1], 'g--', label='SF = 1.25')
plt.legend()
plt.show()

MSE = np.mean(np.square(np.subtract(y_test, y_predict)))
print('MSE = ', MSE)
print('scaled SF = ', 1/0.8)
print('mean SF = ', np.round(np.mean(df['V_test']/df['V_Rd']),2))
print('fitted SF = ', 1.80)

In [ ]:
plt.scatter(df['V_Rd'], df['V_test'])
plt.xlabel('V_predicted_EC')
plt.ylabel('V_test')
plt.plot([0,1.75], [0, 1.75*1.8])
plt.show()
print('SF = 1.8')

In [ ]:
from numpy import set_printoptions
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression, SelectFpr
from matplotlib import pyplot

scaled_data = df_scaled.values


X = scaled_data[:,0:4]
Y = scaled_data[:,4]

# feature extraction
test = SelectKBest(score_func=SelectFpr, k=4)
fit = test.fit(X, Y)


# summarize scores
set_printoptions(precision=3)
print(fit.scores_)

# plot feature importance
pyplot.bar([x for x in range(len(fit.scores_))], fit.scores_)
pyplot.xlabel('Feature $i$')
pyplot.ylabel('Feature Score ANOVA F-value')
pyplot.grid()
pyplot.show()


features = fit.transform(X)
# summarize selected features
print(features[0:5,:])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# select the properties that should be scaled
cols=['d', 'Fläche', 'rho_l', 'fcm,cyl', 'V_test']

# Create Scaler object
scaler = MinMaxScaler(feature_range=(0.000001, 1))
# fit scaler to data
scaler.fit(df[cols])
#scale chosen properties
df_scaled = pd.DataFrame(scaler.transform(df[cols]), columns=cols)

data = df_scaled.values
X = data[:,0:4]
Y = data[:,4]
# feature extraction
test = SelectKBest(score_func=f_regression, k='all')
fit = test.fit(X, Y)

# summarize scores
set_printoptions(precision=3)
SKB = fit.scores_/np.amax(fit.scores_)

features = fit.transform(X)
# summarize selected features
print(SKB)
import sklearn
SP_ = sklearn.feature_selection.SelectPercentile(score_func=f_regression, percentile=100).fit(X,Y).scores_
SP = SP_/np.amax(SP_)

# plot feature importance

fig = plt.figure()
ax = fig.add_axes([0,0,1,1])
ax.bar(np.arange(4) + 0.00, SKB, color = 'b', width = 0.25, label='SelectKBest')
#ax.bar(np.arange(4) + 0.25, SP, color = 'g', width = 0.25, label='SelectPercentile')
plt.legend()
ax.set(xlabel='     d                              Fläche                              rho_l                              fcm_cyl', ylabel='feature importance')
ax.text(-.05,SKB[0]+0.01, SKB[0], color='blue', fontweight='bold')
ax.text(0.85,SKB[1]+0.01, np.round(SKB[1],4), color='blue', fontweight='bold')
ax.text(1.85,SKB[2]+0.01, np.round(SKB[2],4), color='blue', fontweight='bold')
ax.text(2.85,SKB[3]+0.01, np.round(SKB[3],4), color='blue', fontweight='bold')
plt.legend()
plt.show()



In [ ]:
df_scaled.head()